# Angkor's engineered water landscape

A historical reconstruction of Angkor's water network at its mature, peak-operation scale. The network accumulated over centuries: the East Baray dates to the 9th century, the West Baray to the 11th, and Angkor Wat and Angkor Thom to the 12th. This is a cumulative historical landscape, not a claim that every feature operated in the same configuration at one moment.

The map combines current terrain and mapped channel alignments with historically dated reservoirs and monuments. Individual canal dates, exact water levels, field boundaries, and flow paths are not established by the mapped geometry. Southward surplus drainage toward Tonle Sap is shown as a broad interpretive relationship, not as a surveyed route or hydraulic simulation. No precise population, storage, or irrigated-area totals are inferred.

## Sources and reproducibility

- Fletcher, R. et al. (2008), [The water management network of Angkor, Cambodia](https://doi.org/10.1017/S0003598X00097295), *Antiquity*. Describes a network initiated in the 9th century, expanded and modified over time, with surplus drainage toward Tonle Sap; large new channels from the 12th century primarily disposed of water to the lake.
- NASA Earth Observatory, [Angkor Ruins](https://science.nasa.gov/earth/earth-observatory/angkor-ruins-4995/). Provides dates and context for East and West Barays, Angkor Wat, and Angkor Thom.
- Terrain: [SRTM 1-arc-second HGT](https://elevation-tiles-prod.s3.amazonaws.com/skadi/N13/N13E103.hgt.gz), distributed by Mapzen Elevation Tiles; SRTM data courtesy of NASA/USGS.
- Waterway alignments, major-feature geometries, and the present-day Tonle Sap outline: [OpenStreetMap contributors](https://www.openstreetmap.org/copyright), queried via [Overpass API](https://overpass-api.de/); OSM data is licensed under the [ODbL](https://www.openstreetmap.org/copyright). These alignments are not individually dated by this map.
- Terrain and channel snapshots are stored in `data/angkor_dem.npz` and `data/angkor_water_features.geojson` so the plot can be regenerated offline. Set `REFRESH_DATA = True` to request fresh source data.

The plot and its page template are generated below and exported to `output/20261031_angkor_water_system.html`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import gzip
import hashlib
import json
import math
import sys
import urllib.parse
import urllib.request
import numpy as np
import plotly.graph_objects as go

current = Path.cwd().resolve()
project_root = next((p for p in (current, *current.parents)
                     if (p / 'site_export.py').is_file() and (p / 'notebooks').is_dir()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the repository or notebooks directory.')
sys.path.insert(0, str(project_root)) if str(project_root) not in sys.path else None
from site_export import write_plot_page

DATA_DIR = project_root / 'data'
CACHE_DIR = project_root / '.cache' / 'angkor'
DEM_FILE = DATA_DIR / 'angkor_dem.npz'
FEATURE_FILE = DATA_DIR / 'angkor_water_features.geojson'
output_file = project_root / 'output' / '20261031_angkor_water_system.html'
REFRESH_DATA = False

LAT_MIN, LAT_MAX = 13.18, 13.50
LON_MIN, LON_MAX = 103.65, 104.08
CENTER_LAT, CENTER_LON = 13.4125210, 103.8665896
SRTM_TILES = ('N13E103', 'N13E104')


def fetch_bytes(url, *, timeout=90):
    request = urllib.request.Request(url, headers={'User-Agent': 'phey.app/1.0 (https://phey.app)'})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        if response.status != 200:
            raise RuntimeError(f'Data request failed ({response.status}): {url}')
        return response.read()


DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {project_root}')


Project root: C:\Users\PH\Python\p00


In [2]:
# Cache source data so the normal notebook run is reproducible and offline.
if REFRESH_DATA or not DEM_FILE.exists():
    elevation_parts = []
    longitude_parts = []
    tile_hashes = []
    latitude = None
    step = 4
    for tile_name in SRTM_TILES:
        tile_cache = CACHE_DIR / f'{tile_name}.hgt.gz'
        tile_url = f'https://elevation-tiles-prod.s3.amazonaws.com/skadi/{tile_name[:3]}/{tile_name}.hgt.gz'
        if REFRESH_DATA or not tile_cache.exists():
            tile_bytes = fetch_bytes(tile_url, timeout=180)
            tile_cache.write_bytes(tile_bytes)
        else:
            tile_bytes = tile_cache.read_bytes()
        tile_hashes.append(hashlib.sha256(tile_bytes).hexdigest())
        samples = np.frombuffer(gzip.decompress(tile_bytes), dtype='>i2')
        if samples.size != 3601 * 3601:
            raise ValueError(f'Unexpected HGT sample count for {tile_name}: {samples.size}')
        tile = samples.reshape((3601, 3601)).astype(np.int16)
        tile_north = int(tile_name[1:3]) + 1
        tile_west = int(tile_name[4:7])
        tile_lat = tile_north - np.arange(3601, dtype=np.float64) / 3600
        tile_lon = tile_west + np.arange(3601, dtype=np.float64) / 3600
        rows = np.flatnonzero((tile_lat >= LAT_MIN) & (tile_lat <= LAT_MAX))
        cols = np.flatnonzero((tile_lon >= LON_MIN) & (tile_lon <= LON_MAX))
        if not rows.size or not cols.size:
            raise ValueError(f'SRTM tile {tile_name} does not cover the requested Angkor crop.')
        r0, r1 = rows[0], rows[-1] + 1
        c0, c1 = cols[0], cols[-1] + 1
        crop = tile[r0:r1:step, c0:c1:step].astype(np.float32)
        crop[crop == -32768] = np.nan
        elevation_parts.append(crop)
        longitude_parts.append(tile_lon[c0:c1:step])
        tile_latitudes = tile_lat[r0:r1:step]
        if latitude is None:
            latitude = tile_latitudes
        elif not np.array_equal(latitude, tile_latitudes):
            raise ValueError(f'SRTM latitude grids do not align for {tile_name}.')
    elevation = np.concatenate(elevation_parts, axis=1)
    longitude = np.concatenate(longitude_parts)
    if elevation.size < 1000 or not np.isfinite(elevation).any():
        raise ValueError('SRTM crop is empty or contains no valid terrain samples.')
    dem_hash = hashlib.sha256(''.join(sorted(tile_hashes)).encode()).hexdigest()
    np.savez_compressed(DEM_FILE, latitude=latitude, longitude=longitude,
                        elevation=elevation, tile=','.join(SRTM_TILES), sha256=dem_hash,
                        accessed=datetime.now(timezone.utc).date().isoformat())

if REFRESH_DATA or not FEATURE_FILE.exists():
    bbox = f'{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}'
    query = (f'[out:json][timeout:40];way["waterway"~"^(canal|ditch)$"]({bbox});out tags geom;')
    overpass_url = 'https://overpass-api.de/api/interpreter?data=' + urllib.parse.quote(query, safe='')
    osm = json.loads(fetch_bytes(overpass_url, timeout=120))
    features = []
    for element in osm.get('elements', []):
        geometry = element.get('geometry')
        if not geometry or len(geometry) < 2:
            continue
        tags = element.get('tags', {})
        coordinates = [[point['lon'], point['lat']] for point in geometry]
        name = tags.get('name') or tags.get('name:en') or f"{tags.get('waterway', 'waterway').title()} (OSM way {element['id']})"
        features.append({
            'type': 'Feature',
            'geometry': {'type': 'LineString', 'coordinates': coordinates},
            'properties': {'name': name, 'class': tags.get('waterway', 'waterway'),
                           'osm_type': 'way', 'osm_id': element['id'],
                           'source_url': f"https://www.openstreetmap.org/way/{element['id']}"}
        })

    major_relations = {
        '6983650': 'West Baray',
        '6647487': 'East Baray (Thnal Baray)',
        '13337382': 'Jayatataka Baray',
        '13276237': 'Angkor Thom moat',
        '5638825': 'Tonle Sap (modern seasonal extent)',
    }
    ids = ','.join('R' + osm_id for osm_id in major_relations)
    lookup_url = 'https://nominatim.openstreetmap.org/lookup?' + urllib.parse.urlencode({
        'format': 'jsonv2', 'polygon_geojson': 1, 'osm_ids': ids
    })
    relations = json.loads(fetch_bytes(lookup_url, timeout=60))
    found = {str(item.get('osm_id')): item for item in relations if item.get('geojson')}
    missing = set(major_relations) - set(found)
    if missing:
        raise ValueError(f'OpenStreetMap geometry missing for major feature ids: {sorted(missing)}')
    for osm_id, label in major_relations.items():
        item = found[osm_id]
        features.append({
            'type': 'Feature', 'geometry': item['geojson'],
            'properties': {'name': label, 'class': 'major water feature',
                           'osm_type': 'relation', 'osm_id': int(osm_id),
                           'source_url': f'https://www.openstreetmap.org/relation/{osm_id}'}
        })

    collection = {
        'type': 'FeatureCollection',
        'metadata': {'source': 'OpenStreetMap contributors', 'accessed': datetime.now(timezone.utc).date().isoformat(),
                     'license': 'ODbL', 'query_bbox': bbox, 'overpass_query': query,
                     'feature_count': len(features)},
        'features': features,
    }
    FEATURE_FILE.write_text(json.dumps(collection, ensure_ascii=False, separators=(',', ':')), encoding='utf-8')

with np.load(DEM_FILE, allow_pickle=False) as dem:
    latitude = dem['latitude']
    longitude = dem['longitude']
    elevation = dem['elevation']
    dem_hash = str(dem['sha256'])
feature_collection = json.loads(FEATURE_FILE.read_text(encoding='utf-8'))
features = feature_collection['features']
if elevation.shape != (latitude.size, longitude.size) or elevation.size == 0:
    raise ValueError('Terrain arrays have inconsistent dimensions.')
if not features:
    raise ValueError('No mapped water features were loaded.')
print(f'Terrain grid: {elevation.shape[1]} x {elevation.shape[0]} samples; SRTM SHA-256 {dem_hash}')
print(f'Water feature snapshot: {len(features)} features, accessed {feature_collection["metadata"]["accessed"]}')


Terrain grid: 389 x 289 samples; SRTM SHA-256 50f851841400ae46f141401022ac2710bb72b020de9f2674a67149523c835c00
Water feature snapshot: 257 features, accessed 2026-10-31


In [3]:
# Convert the terrain grid to a local kilometre grid for a correctly proportioned regional map.
km_per_degree_lat = 111.195
km_per_degree_lon = km_per_degree_lat * math.cos(math.radians(CENTER_LAT))
x_km = (longitude - CENTER_LON) * km_per_degree_lon
y_km = (latitude - CENTER_LAT) * km_per_degree_lat

fig = go.Figure()
fig.add_trace(go.Heatmap(
    x=x_km, y=y_km, z=elevation,
    colorscale=[[0, '#315a57'], [0.20, '#58775f'], [0.42, '#92916b'],
                [0.68, '#b8a67c'], [0.88, '#d9cfb4'], [1, '#f1eadb']],
    colorbar={'title': {'text': 'Elevation (m)'}, 'thickness': 12, 'len': 0.68,
              'outlinewidth': 0, 'x': 1.02},
    hovertemplate='Terrain: %{z:.0f} m<extra></extra>',
    name='SRTM terrain', showscale=True, zsmooth='best',
))
fig.add_trace(go.Contour(
    x=x_km, y=y_km, z=elevation, showscale=False,
    contours={'start': 20, 'end': 100, 'size': 20, 'coloring': 'none',
              'showlabels': True, 'labelfont': {'size': 9, 'color': '#39413b'}},
    line={'color': 'rgba(42, 51, 45, 0.38)', 'width': 0.7},
    name='20 m contours', hoverinfo='skip',
))

canal_x, canal_y, canal_text = [], [], []
for feature in features:
    properties = feature.get('properties', {})
    geometry = feature.get('geometry') or {}
    geom_type = geometry.get('type')
    coordinates = geometry.get('coordinates', [])
    if properties.get('class') in {'canal', 'ditch'} and geom_type == 'LineString':
        if len(coordinates) < 2:
            continue
        for lon, lat in coordinates:
            canal_x.append((lon - CENTER_LON) * km_per_degree_lon)
            canal_y.append((lat - CENTER_LAT) * km_per_degree_lat)
            canal_text.append(properties.get('name', 'Mapped waterway'))
        canal_x.append(None); canal_y.append(None); canal_text.append(None)
if canal_x:
    fig.add_trace(go.Scatter(
        x=canal_x, y=canal_y, text=canal_text, mode='lines',
        line={'color': '#087fbd', 'width': 1.4}, opacity=0.76,
        name='Mapped channels (date/function uncertain)', legendgroup='canals',
        hovertemplate='%{text}<extra>OpenStreetMap</extra>',
    ))


def coordinate_lines(geometry):
    kind = geometry.get('type')
    coords = geometry.get('coordinates', [])
    if kind == 'LineString':
        return [coords]
    if kind == 'MultiLineString':
        return coords
    if kind == 'Polygon':
        return coords[:1]
    if kind == 'MultiPolygon':
        return [ring for polygon in coords for ring in polygon[:1]]
    return []


def to_xy(coords):
    return ([(point[0] - CENTER_LON) * km_per_degree_lon for point in coords],
            [(point[1] - CENTER_LAT) * km_per_degree_lat for point in coords])

historical_labels = {
    'West Baray': 'West Baray | 11th c.',
    'East Baray (Thnal Baray)': 'East Baray | 9th c.',
    'Jayatataka Baray': 'Jayatataka Baray | late 12th c.',
    'Angkor Thom moat': 'Angkor Thom moat | 12th c.',
    'Tonle Sap (modern seasonal extent)': 'Tonle Sap | modern seasonal outline',
}

for feature in features:
    properties = feature.get('properties', {})
    if properties.get('class') != 'major water feature':
        continue
    feature_name = properties['name']
    display_name = historical_labels.get(feature_name, feature_name)
    geometry = feature.get('geometry') or {}
    parts = coordinate_lines(geometry)
    x_parts, y_parts = [], []
    for part in parts:
        if len(part) < 2:
            continue
        px, py = to_xy(part)
        x_parts.extend(px); y_parts.extend(py)
        x_parts.append(None); y_parts.append(None)
    if not x_parts:
        continue
    closed = geometry.get('type') in {'Polygon', 'MultiPolygon'}
    fig.add_trace(go.Scatter(
        x=x_parts, y=y_parts, mode='lines', fill='toself' if closed else None,
        fillcolor='rgba(20, 145, 190, 0.34)' if closed else None,
        line={'color': '#087fbd', 'width': 2.2},
        name=display_name, legendgroup='major-water',
        hovertemplate=f"{display_name}<br>OSM relation {properties['osm_id']}<extra></extra>",
    ))

angkor_wat_lon, angkor_wat_lat = 103.8665896, 13.4125210
fig.add_trace(go.Scatter(
    x=[(angkor_wat_lon - CENTER_LON) * km_per_degree_lon],
    y=[(angkor_wat_lat - CENTER_LAT) * km_per_degree_lat],
    mode='markers+text', text=['Angkor Wat | early 12th c.'], textposition='bottom right',
    marker={'size': 9, 'color': '#f5bc45', 'line': {'color': '#3b3020', 'width': 1.2}},
    name='Angkor Wat', hovertemplate='Angkor Wat | early 12th c.<br>OpenStreetMap way 1113380295<extra></extra>',
))

fig.update_layout(
    title={'text': 'Angkor-to-Tonle-Sap corridor | cumulative system, 9th-13th c.', 'x': 0.02, 'xanchor': 'left'},
    template='plotly_white',
    paper_bgcolor='#f4f1e8', plot_bgcolor='#d9e3dd',
    margin={'l': 54, 'r': 92, 't': 72, 'b': 58},
    width=1200, height=980,
    hovermode='closest',
    legend={'orientation': 'h', 'y': -0.08, 'x': 0, 'groupclick': 'togglegroup'},
    xaxis={'title': 'Kilometres east of Angkor Wat', 'zeroline': False, 'showgrid': False,
           'range': [(LON_MIN - CENTER_LON) * km_per_degree_lon, (LON_MAX - CENTER_LON) * km_per_degree_lon],
           'scaleanchor': 'y', 'scaleratio': 1},
    yaxis={'title': 'Kilometres north of Angkor Wat', 'zeroline': False, 'showgrid': False,
           'range': [(LAT_MIN - CENTER_LAT) * km_per_degree_lat, (LAT_MAX - CENTER_LAT) * km_per_degree_lat]},
    annotations=[
        {'text': 'N', 'x': 0.985, 'y': 0.98, 'xref': 'paper', 'yref': 'paper',
         'showarrow': True, 'ax': 0, 'ay': 32, 'arrowhead': 2,
         'font': {'size': 14, 'color': '#263b45'}},
        {'text': 'Surplus drainage south | schematic',
         'x': 0.5, 'y': 0.025, 'xref': 'paper', 'yref': 'paper', 'showarrow': False,
         'xanchor': 'center', 'font': {'size': 11, 'color': '#17445a'},
         'bgcolor': 'rgba(244,241,232,0.86)', 'borderpad': 4},
    ],
)
fig.show()


This is a historically framed, cumulative reconstruction?not a single-date hydraulic model. The East Baray (9th c.), West Baray (11th c.), and major 12th-century works were built in different phases. Fletcher et al. describe a network with additions and changing roles: earlier channels both distributed and disposed water, while large channels added from the 12th century primarily drained toward Tonle Sap. The southward note represents that broad interpretation; it does not assign flow direction to individual blue lines.

Blue lines are present-day OpenStreetMap alignments whose individual dates and functions are not established here. SRTM terrain is not a canal-bed survey; the Tonle Sap outline is a present-day, seasonally variable OSM geometry, not an ancient shoreline; field boundaries, operating levels, and seasonal flows are not reconstructed. See the cited sources above for the historical basis.

In [4]:
plot_html = fig.to_html(
    include_plotlyjs=False, full_html=False, div_id='angkor-water-map',
    config={'responsive': True, 'displayModeBar': False, 'scrollZoom': True},
)
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Angkor-to-Tonle-Sap corridor | cumulative system, 9th-13th c.</title>
  <meta name="description" content="A historically framed map of Angkor waterworks accumulated from the 9th to 13th centuries, with dates and interpretive limits.">
  <style>
    :root {{ color-scheme: dark; font-family: system-ui, sans-serif; background: #101927; color: #f3f4f0; }}
    * {{ box-sizing: border-box; }} body {{ max-width: 1280px; margin: 0 auto; padding: 1.5rem clamp(1rem, 3vw, 2.5rem) 2rem; }}
    a {{ color: #a6d9eb; }}
    h1 {{ margin: .25rem 0 .75rem; font: 400 clamp(2rem, 5vw, 3.4rem)/1.08 Georgia, serif; }}
    .intro, .note {{ max-width: 860px; color: #c4cfda; line-height: 1.65; }} .plot {{ margin: 1.4rem 0; overflow-x: auto; background: #f4f1e8; }}
    .source {{ border-top: 1px solid #ffffff2a; padding-top: 1rem; font-size: .9rem; }} .source p {{ line-height: 1.6; }}
    @media (max-width: 700px) {{ body {{ padding-inline: .5rem; }} .plot {{ margin-inline: -.5rem; }} }}
  </style>
</head>
<body>
  <header><h1>Angkor's water network at peak operation</h1>
    <p class="intro">A cumulative historical landscape built up over centuries, with dated reservoirs and monuments set against mapped channel alignments. Toggle layers and hover over features to explore the system.</p>
  </header>
  <main><div class="plot" role="region" aria-label="Historically framed map of Angkor waterworks">{plot_html}</div>
    <p class="note"><strong>Historical interpretation:</strong> the East Baray (9th c.), West Baray (11th c.), and major 12th-century works belong to different phases of a network that accumulated over time. The southward drainage note is a broad interpretation from the published network study, not a surveyed route; it does not assign flow to individual channels. OSM alignments are not individually dated, and the Tonle Sap outline is a present-day, seasonally variable OSM geometry, not an ancient shoreline; field boundaries, water levels, and seasonal flows are not reconstructed.</p>
    <div class="source"><p>Historical context: <a href="https://doi.org/10.1017/S0003598X00097295">Fletcher et al. (2008), <em>The water management network of Angkor, Cambodia</em></a>; <a href="https://science.nasa.gov/earth/earth-observatory/angkor-ruins-4995/">NASA Earth Observatory, Angkor Ruins</a>. Terrain: NASA/USGS SRTM 1-arc-second data distributed by Mapzen Elevation Tiles. Waterway geometry: <a href="https://www.openstreetmap.org/copyright">OpenStreetMap contributors (ODbL)</a>; queried via Overpass API on {feature_collection['metadata']['accessed']}. OSM alignments are shown for geographic context, not as independently dated archaeological evidence.</p></div>
  </main>
</body>
</html>"""
write_plot_page(output_file, html)
print(f'Wrote {output_file.relative_to(project_root)} ({output_file.stat().st_size:,} bytes)')


Wrote output\20261031_angkor_water_system.html (1,489,175 bytes)
